# TensorFlow tf.data API 最佳實踐

<br>
<a href="https://colab.research.google.com/github/markl-a/My-AI-Learning-Notes/blob/main/1.從AI到LLM基礎/4.DL/01.Tensorflow2/5.TF_Data_Best_Practices.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<br>

## 📚 學習目標

本教程將幫助你掌握：

1. ✅ tf.data.Dataset 基礎操作
2. ✅ 高效數據載入技巧（prefetch, cache, interleave）
3. ✅ 數據增強與轉換
4. ✅ 處理不同數據源（圖像、文本、CSV）
5. ✅ 性能優化最佳實踐

---

## 為什麼使用 tf.data？

### 傳統方法的問題

```python
# ❌ 低效的數據載入方式
for epoch in range(num_epochs):
    for i in range(0, len(data), batch_size):
        batch_data = data[i:i+batch_size]  # 同步載入，GPU 等待
        model.train_on_batch(batch_data)
```

### tf.data 的優勢

- ⚡ **並行處理：** CPU 預處理與 GPU 訓練同時進行
- 🔄 **自動優化：** 使用 AUTOTUNE 自動調整參數
- 💾 **記憶體效率：** 串流處理大型數據集
- 🎯 **簡潔 API：** 鏈式調用，代碼清晰

---

In [ ]:
# 環境設置
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
from tensorflow import keras

print(f"TensorFlow 版本: {tf.__version__}")
print(f"GPU 可用: {tf.config.list_physical_devices('GPU')}")

# 設置隨機種子以確保可重現性
tf.random.set_seed(42)
np.random.seed(42)

## 1. tf.data.Dataset 基礎

### 1.1 創建 Dataset 的多種方式

In [ ]:
# 方法 1: 從 NumPy 數組創建
x_train = np.random.rand(1000, 28, 28, 1).astype(np.float32)
y_train = np.random.randint(0, 10, 1000).astype(np.int32)

dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
print("從 NumPy 創建:", dataset)

# 方法 2: 從生成器創建
def data_generator():
    for i in range(100):
        yield i, i**2

dataset_gen = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=(
        tf.TensorSpec(shape=(), dtype=tf.int32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
)
print("\n從生成器創建:", dataset_gen)

# 方法 3: 從文件路徑創建
file_pattern = "*.jpg"  # 示例
# dataset_files = tf.data.Dataset.list_files(file_pattern)

# 方法 4: 從範圍創建
dataset_range = tf.data.Dataset.range(10)
print("\n從範圍創建:", dataset_range)

### 1.2 基本轉換操作

In [ ]:
# 創建示例數據集
dataset = tf.data.Dataset.range(10)

# 1. map: 應用函數到每個元素
dataset_squared = dataset.map(lambda x: x**2)
print("平方運算:", list(dataset_squared.as_numpy_iterator())[:5])

# 2. filter: 過濾元素
dataset_filtered = dataset.filter(lambda x: x % 2 == 0)
print("過濾偶數:", list(dataset_filtered.as_numpy_iterator()))

# 3. batch: 批次處理
dataset_batched = dataset.batch(3)
print("\n批次處理 (batch_size=3):")
for batch in dataset_batched.take(3):
    print(batch.numpy())

# 4. shuffle: 打亂順序
dataset_shuffled = dataset.shuffle(buffer_size=10)
print("\n打亂順序:", list(dataset_shuffled.as_numpy_iterator()))

# 5. repeat: 重複數據集
dataset_repeated = dataset.take(3).repeat(2)
print("\n重複 2 次:", list(dataset_repeated.as_numpy_iterator()))

## 2. 性能優化核心技術

### 2.1 Prefetch - 數據預取

In [ ]:
# 模擬數據處理和訓練函數
def preprocess(x):
    time.sleep(0.01)  # 模擬預處理時間
    return x

def train_step(x):
    time.sleep(0.01)  # 模擬訓練時間
    return x

# ❌ 沒有 prefetch
dataset_no_prefetch = tf.data.Dataset.range(10)
dataset_no_prefetch = dataset_no_prefetch.map(lambda x: tf.py_function(preprocess, [x], tf.int64))

start = time.time()
for x in dataset_no_prefetch:
    train_step(x)
time_no_prefetch = time.time() - start

# ✅ 使用 prefetch
AUTOTUNE = tf.data.AUTOTUNE
dataset_with_prefetch = tf.data.Dataset.range(10)
dataset_with_prefetch = dataset_with_prefetch.map(
    lambda x: tf.py_function(preprocess, [x], tf.int64)
).prefetch(AUTOTUNE)

start = time.time()
for x in dataset_with_prefetch:
    train_step(x)
time_with_prefetch = time.time() - start

print(f"沒有 prefetch: {time_no_prefetch:.2f} 秒")
print(f"使用 prefetch: {time_with_prefetch:.2f} 秒")
print(f"加速: {time_no_prefetch/time_with_prefetch:.2f}x")

### 2.2 Cache - 數據快取

In [ ]:
# 創建一個需要昂貴計算的數據集
def expensive_preprocessing(x):
    time.sleep(0.001)  # 模擬昂貴的計算
    return x * x

dataset = tf.data.Dataset.range(1000)

# ❌ 沒有 cache - 每個 epoch 都重新計算
dataset_no_cache = dataset.map(
    lambda x: tf.py_function(expensive_preprocessing, [x], tf.int64)
)

start = time.time()
for epoch in range(3):
    for _ in dataset_no_cache:
        pass
time_no_cache = time.time() - start

# ✅ 使用 cache - 第一個 epoch 後快取結果
dataset_with_cache = dataset.map(
    lambda x: tf.py_function(expensive_preprocessing, [x], tf.int64)
).cache()  # 快取到記憶體

start = time.time()
for epoch in range(3):
    for _ in dataset_with_cache:
        pass
time_with_cache = time.time() - start

print(f"沒有 cache (3 epochs): {time_no_cache:.2f} 秒")
print(f"使用 cache (3 epochs): {time_with_cache:.2f} 秒")
print(f"加速: {time_no_cache/time_with_cache:.2f}x")

# 也可以快取到磁盤
# dataset_with_disk_cache = dataset.cache('/tmp/cache_file')

### 2.3 並行 Map - Parallel Mapping

In [ ]:
# 創建較大的數據集
dataset = tf.data.Dataset.range(1000)

def complex_preprocessing(x):
    # 模擬複雜的預處理
    return tf.math.square(x) + tf.math.sqrt(tf.cast(x, tf.float32))

# ❌ 串行 map
start = time.time()
dataset_serial = dataset.map(complex_preprocessing)
for _ in dataset_serial:
    pass
time_serial = time.time() - start

# ✅ 並行 map
start = time.time()
dataset_parallel = dataset.map(
    complex_preprocessing,
    num_parallel_calls=AUTOTUNE  # 自動調整並行數量
)
for _ in dataset_parallel:
    pass
time_parallel = time.time() - start

print(f"串行 map: {time_serial:.2f} 秒")
print(f"並行 map: {time_parallel:.2f} 秒")
print(f"加速: {time_serial/time_parallel:.2f}x")

## 3. 完整的數據管道範例

### 3.1 圖像數據管道 - MNIST

In [ ]:
# 載入 MNIST 數據集
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 數據預處理函數
def preprocess_image(image, label):
    # 正規化到 [0, 1]
    image = tf.cast(image, tf.float32) / 255.0
    # 添加通道維度
    image = tf.expand_dims(image, -1)
    return image, label

# 數據增強函數
def augment_image(image, label):
    # 隨機旋轉
    image = tf.image.rot90(image, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    # 隨機翻轉（對某些數據集有用）
    # image = tf.image.random_flip_left_right(image)
    return image, label

# 建立訓練數據管道
BATCH_SIZE = 128
BUFFER_SIZE = 10000

train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .cache()  # 在預處理後快取
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .map(augment_image, num_parallel_calls=AUTOTUNE)  # 批次級數據增強
    .prefetch(AUTOTUNE)
)

# 建立測試數據管道（不需要增強和打亂）
test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(AUTOTUNE)
)

print(f"訓練數據集: {train_dataset}")
print(f"測試數據集: {test_dataset}")

# 視覺化一個批次
for images, labels in train_dataset.take(1):
    print(f"批次形狀: images={images.shape}, labels={labels.shape}")
    
    plt.figure(figsize=(10, 10))
    for i in range(min(25, BATCH_SIZE)):
        plt.subplot(5, 5, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap='gray')
        plt.title(f"Label: {labels[i].numpy()}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

### 3.2 使用優化管道訓練模型

In [ ]:
# 建立簡單的 CNN 模型
model = keras.Sequential([
    keras.layers.Conv2D(32, 3, activation='relu', input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 訓練模型
history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=test_dataset,
    verbose=1
)

# 繪製訓練歷史
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='訓練準確率')
plt.plot(history.history['val_accuracy'], label='驗證準確率')
plt.xlabel('Epoch')
plt.ylabel('準確率')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='訓練損失')
plt.plot(history.history['val_loss'], label='驗證損失')
plt.xlabel('Epoch')
plt.ylabel('損失')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 4. 處理不同數據源

### 4.1 從 CSV 文件載入

In [ ]:
# 創建示例 CSV 數據
import pandas as pd

# 生成示例數據
data = {
    'feature1': np.random.rand(1000),
    'feature2': np.random.rand(1000),
    'feature3': np.random.rand(1000),
    'label': np.random.randint(0, 2, 1000)
}
df = pd.DataFrame(data)
df.to_csv('/tmp/example_data.csv', index=False)

# 使用 tf.data 載入 CSV
def get_dataset_from_csv(file_path, batch_size=32):
    # 定義列名和默認值
    column_names = ['feature1', 'feature2', 'feature3', 'label']
    label_name = 'label'
    
    # 創建數據集
    dataset = tf.data.experimental.make_csv_dataset(
        file_path,
        batch_size=batch_size,
        label_name=label_name,
        num_epochs=1,
        shuffle=True,
        shuffle_buffer_size=10000
    )
    
    return dataset

csv_dataset = get_dataset_from_csv('/tmp/example_data.csv')

# 檢視數據
for features, labels in csv_dataset.take(1):
    print("特徵:")
    for key, value in features.items():
        print(f"  {key}: {value[:5].numpy()}")
    print(f"標籤: {labels[:5].numpy()}")

### 4.2 從圖像文件載入

In [ ]:
# 模擬圖像文件路徑（實際使用時替換為真實路徑）
def create_image_dataset_from_directory(directory, image_size=(224, 224), batch_size=32):
    """
    從目錄載入圖像數據集
    
    目錄結構應為:
    directory/
        class1/
            image1.jpg
            image2.jpg
        class2/
            image3.jpg
            image4.jpg
    """
    dataset = keras.utils.image_dataset_from_directory(
        directory,
        image_size=image_size,
        batch_size=batch_size,
        label_mode='int'  # 'int', 'categorical', 'binary' 或 None
    )
    
    # 正規化
    normalization_layer = keras.layers.Rescaling(1./255)
    dataset = dataset.map(lambda x, y: (normalization_layer(x), y))
    
    # 優化性能
    dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)
    
    return dataset

# 使用示例（註解掉，因為需要實際圖像目錄）
# train_dataset = create_image_dataset_from_directory(
#     'path/to/train_directory',
#     image_size=(224, 224),
#     batch_size=32
# )

print("圖像載入函數已定義")

### 4.3 文本數據處理

In [ ]:
from tensorflow.keras.layers import TextVectorization

# 示例文本數據
texts = [
    "TensorFlow is awesome",
    "Deep learning is fun",
    "I love machine learning",
    "Neural networks are powerful",
    "Data science is interesting"
] * 200  # 重複以增加數據量

labels = [1, 0, 1, 1, 0] * 200

# 創建文本向量化層
max_tokens = 1000
sequence_length = 10

vectorize_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

# 適應詞彙表
vectorize_layer.adapt(texts)

# 創建數據集
text_dataset = tf.data.Dataset.from_tensor_slices((texts, labels))

# 向量化文本
def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    return vectorize_layer(text), label

text_dataset = (
    text_dataset
    .map(vectorize_text, num_parallel_calls=AUTOTUNE)
    .batch(32)
    .cache()
    .prefetch(AUTOTUNE)
)

# 查看詞彙表
vocab = vectorize_layer.get_vocabulary()
print(f"詞彙表大小: {len(vocab)}")
print(f"前 20 個詞: {vocab[:20]}")

# 查看向量化結果
for text_batch, label_batch in text_dataset.take(1):
    print(f"\n文本批次形狀: {text_batch.shape}")
    print(f"標籤批次形狀: {label_batch.shape}")
    print(f"第一個樣本: {text_batch[0].numpy()}")

## 5. 高級技巧

### 5.1 自定義數據生成器

In [ ]:
class CustomDataGenerator:
    """自定義數據生成器範例"""
    
    def __init__(self, data, labels, batch_size=32):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.indices = np.arange(len(data))
        
    def __call__(self):
        """生成器函數"""
        np.random.shuffle(self.indices)
        
        for start_idx in range(0, len(self.data), self.batch_size):
            end_idx = min(start_idx + self.batch_size, len(self.data))
            batch_indices = self.indices[start_idx:end_idx]
            
            batch_data = self.data[batch_indices]
            batch_labels = self.labels[batch_indices]
            
            # 可以在這裡添加自定義的數據增強
            batch_data = batch_data + np.random.normal(0, 0.01, batch_data.shape)
            
            yield batch_data.astype(np.float32), batch_labels.astype(np.int32)

# 使用自定義生成器
data = np.random.rand(1000, 10)
labels = np.random.randint(0, 2, 1000)

generator = CustomDataGenerator(data, labels, batch_size=32)

dataset = tf.data.Dataset.from_generator(
    generator,
    output_signature=(
        tf.TensorSpec(shape=(None, 10), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)
    )
)

dataset = dataset.prefetch(AUTOTUNE)

# 測試
for batch_data, batch_labels in dataset.take(2):
    print(f"批次數據形狀: {batch_data.shape}, 標籤形狀: {batch_labels.shape}")

### 5.2 混合數據源

In [ ]:
# 創建兩個不同的數據集
dataset1 = tf.data.Dataset.range(0, 100).map(lambda x: x)
dataset2 = tf.data.Dataset.range(100, 200).map(lambda x: x)

# 方法 1: 連接數據集
concatenated = dataset1.concatenate(dataset2)
print("連接數據集:", list(concatenated.take(5).as_numpy_iterator()))

# 方法 2: 交錯數據集
interleaved = tf.data.Dataset.zip((dataset1, dataset2))
print("\n交錯數據集:")
for d1, d2 in interleaved.take(3):
    print(f"Dataset1: {d1.numpy()}, Dataset2: {d2.numpy()}")

# 方法 3: 採樣數據集（按比例混合）
dataset1_sampled = dataset1.repeat()
dataset2_sampled = dataset2.repeat()

# 按 70:30 的比例混合
choice_dataset = tf.data.Dataset.range(2).repeat()
mixed_dataset = tf.data.Dataset.choose_from_datasets(
    [dataset1_sampled, dataset2_sampled],
    choice_dataset
)

print("\n混合數據集:", list(mixed_dataset.take(10).as_numpy_iterator()))

### 5.3 動態批次大小（Bucketing）

In [ ]:
# 創建不同長度的序列
sequences = [
    tf.constant([1, 2, 3]),
    tf.constant([4, 5]),
    tf.constant([6, 7, 8, 9]),
    tf.constant([10]),
    tf.constant([11, 12, 13, 14, 15]),
]

# 將相似長度的序列分組到同一批次
def length_fn(x):
    return tf.shape(x)[0]

# 定義桶邊界和批次大小
bucket_boundaries = [2, 4, 6]
bucket_batch_sizes = [2, 2, 2, 2]  # 每個桶的批次大小

dataset = tf.data.Dataset.from_tensor_slices(sequences)
bucketed_dataset = dataset.bucket_by_sequence_length(
    element_length_func=length_fn,
    bucket_boundaries=bucket_boundaries,
    bucket_batch_sizes=bucket_batch_sizes,
    padded_shapes=None,  # 自動填充
    padding_values=0
)

print("分桶批次處理:")
for i, batch in enumerate(bucketed_dataset.take(3)):
    print(f"\n批次 {i+1}:")
    print(batch.numpy())

## 6. 性能基準測試

### 6.1 比較不同優化策略的性能

In [ ]:
def benchmark_dataset(dataset, num_epochs=2, name="Dataset"):
    """測試數據集性能"""
    start_time = time.time()
    
    for epoch in range(num_epochs):
        for _ in dataset:
            pass
    
    total_time = time.time() - start_time
    print(f"{name}: {total_time:.2f} 秒")
    return total_time

# 準備數據
(x_train, y_train), _ = keras.datasets.mnist.load_data()

# 1. 基礎管道（無優化）
ds_basic = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .batch(128)
)

# 2. 添加 prefetch
ds_prefetch = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .batch(128)
    .prefetch(AUTOTUNE)
)

# 3. 添加 cache
ds_cache = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .cache()
    .batch(128)
    .prefetch(AUTOTUNE)
)

# 4. 完整優化
ds_optimized = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(10000)
    .batch(128)
    .prefetch(AUTOTUNE)
)

# 基準測試
print("性能基準測試 (2 epochs):")
print("=" * 40)
time_basic = benchmark_dataset(ds_basic, name="1. 基礎管道")
time_prefetch = benchmark_dataset(ds_prefetch, name="2. + Prefetch")
time_cache = benchmark_dataset(ds_cache, name="3. + Cache")
time_optimized = benchmark_dataset(ds_optimized, name="4. 完整優化")

# 可視化結果
plt.figure(figsize=(10, 6))
strategies = ['基礎', '+ Prefetch', '+ Cache', '完整優化']
times = [time_basic, time_prefetch, time_cache, time_optimized]
colors = ['red', 'orange', 'yellow', 'green']

bars = plt.bar(strategies, times, color=colors, alpha=0.7, edgecolor='black')
plt.ylabel('時間 (秒)', fontsize=12)
plt.title('不同優化策略的性能比較', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# 添加數值標籤
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{time_val:.2f}s\n({time_basic/time_val:.1f}x)',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 7. 最佳實踐總結

### ✅ 推薦的數據管道模板

```python
# 訓練數據集
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)  # 1. 預處理
    .cache()  # 2. 快取（如果數據集適合記憶體）
    .shuffle(buffer_size=10000)  # 3. 打亂
    .batch(batch_size)  # 4. 批次處理
    .map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)  # 5. 批次級增強
    .prefetch(tf.data.AUTOTUNE)  # 6. 預取
)

# 驗證/測試數據集
val_dataset = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .cache()  # 驗證集不需要打亂和增強
    .prefetch(tf.data.AUTOTUNE)
)
```

### 📋 優化檢查清單

- ✅ **使用 `num_parallel_calls=AUTOTUNE`** 進行並行數據處理
- ✅ **適當使用 `.cache()`** - 數據集小時快取到記憶體，大時快取到磁盤
- ✅ **總是使用 `.prefetch(AUTOTUNE)`** 作為最後一步
- ✅ **在 `.batch()` 之前進行 `.shuffle()`**
- ✅ **數據增強在批次級別進行** 可以更高效
- ✅ **使用 `.repeat()` 時要小心** - 確保知道數據集會重複多少次
- ✅ **監控 CPU 和 GPU 使用率** - 找出瓶頸

### ⚠️ 常見錯誤

1. **❌ 在 cache 之後 shuffle**
   ```python
   # 錯誤
   dataset.cache().shuffle(1000)  # shuffle 只影響快取的數據
   
   # 正確
   dataset.shuffle(1000).cache()  # 每個 epoch 都打亂
   ```

2. **❌ 過大的 shuffle buffer**
   ```python
   # 如果數據集有 100 萬樣本，不需要這麼大的 buffer
   dataset.shuffle(1000000)  # 可能導致記憶體問題
   
   # 通常 10000-50000 就足夠了
   dataset.shuffle(10000)
   ```

3. **❌ 忘記 prefetch**
   ```python
   # 缺少 prefetch 會導致 GPU 空閒時間
   dataset.batch(32)  # ❌
   
   dataset.batch(32).prefetch(AUTOTUNE)  # ✅
   ```

---

## 🎓 進一步學習

- [TensorFlow 官方數據性能指南](https://www.tensorflow.org/guide/data_performance)
- [tf.data API 文檔](https://www.tensorflow.org/api_docs/python/tf/data)
- [TensorFlow Datasets](https://www.tensorflow.org/datasets) - 預處理好的常用數據集

---

## 📝 練習建議

1. 使用 tf.data 實作 CIFAR-10 圖像分類管道
2. 比較不同優化策略對訓練時間的影響
3. 實作自定義數據增強管道
4. 處理大型圖像數據集（使用 TFRecord）
5. 實作多模態數據管道（圖像 + 文本）